# Capstone Langlands — l'escalier : de la forme calculée au lac qui la démontre

**Navigation** : [01 — Formes modulaires SL2Z et Hecke](Langlands/01-formes-modulaires-sl2z-hecke.ipynb) · [02 — Monstrous Moonshine et l'invariant j](Langlands/02-monstrous-moonshine-invariant-j.ipynb) · [README de la sous-série](Langlands/README.md) · [lac `hecke_lean`](hecke_lean/README.md) · [Lean-29 — compagnon natif Hecke](Lean-29-Hecke-Operators-Native.ipynb)

La sous-série **Langlands** regarde les formes modulaires comme un pont : de
l'analyse complexe (le demi-plan supérieur, l'action de SL₂(ℤ)) vers l'arithmétique
(les coefficients de Fourier, les valeurs propres de Hecke) puis vers l'algèbre
(Moonshine : le groupe Monstre qui apparaît dans les coefficients de j). Le
programme de Langlands est le nom général de ces correspondances ; la série mère
l'avait croisé à travers son compagnon natif [Lean-29](Lean-29-Hecke-Operators-Native.ipynb),
la sous-série [Langlands](Langlands/README.md) le traite en profondeur.

Cet escalier en est la porte d'entrée depuis la série mère : il part du calcul —
la fonction Δ de Ramanujan, coefficient par coefficient, en arithmétique entière
exacte — monte par la relation de Hecke vérifiée sur la table, montre un témoin
qui la viole, puis lit le lac [`hecke_lean`](hecke_lean/) comme un consommateur :
le fichier réel, sa définition `coeffHeckeT`, la preuve et son audit. La dernière
marche traverse le pont Moonshine. Aucune étape ne demande un kernel Lean : tout
est calculable en Python stdlib, et chaque identité est **calculée dans une sortie**,
comme le README de la sous-série s'y engage.

## 0. Le problème : une identité arithmétique, quatre garanties

Le fil rouge de cet escalier est **une seule identité** — la relation de Hecke sur
les coefficients de Fourier d'une forme modulaire propre :

$$a(p\,n) + p^{k-1}\,a(n/p) = a(p)\,a(n) \qquad (p \text{ premier},\ k = 12)$$

où $a(n) = \tau(n)$, la fonction tau de Ramanujan, coefficient de $q^n$ dans le
produit d'Euler $\Delta = q\prod_{m \geq 1}(1-q^m)^{24}$.

Le même énoncé peut être « vérifié » à quatre niveaux de garantie, et cet escalier
les monte tous : (1) un exemple calculé sur une table finie, (2) l'identité
vérifiée premier par premier — le niveau auquel la relation **caractérise** la
suite, témoin négatif à l'appui, (3) l'énoncé dans le lac `hecke_lean` — la
définition `coeffHeckeT` qui formalise le membre de gauche, (4) la preuve Lean
et son audit d'axiomes. La sous-série ajoute le pont final : Moonshine, où les
coefficients d'une autre forme modulaire (l'invariant $j$) encodent les dimensions
des représentations du groupe Monstre.

## 1. Marche 1 — Δ calculée, coefficient par coefficient

*(double [Langlands-01](Langlands/01-formes-modulaires-sl2z-hecke.ipynb), section produit d'Euler)*

Le produit d'Euler $\prod_{m}(1-q^m)^{24}$ se développe par convolution entière :
aucun flottant, aucune bibliothèque — la série tronquée à l'ordre $N$ est un
polynôme en $q$ à coefficients entiers, et $\tau(n)$ est son coefficient de $q^n$.

In [1]:
# Marche 1 : la fonction tau de Ramanujan par convolution entiere du produit d'Euler.
def tau_table(N):
    """Coefficients tau(1..N) via prod_{m>=1} (1-q^m)^24, arithmetique entiere exacte."""
    poly = [1] + [0] * N          # poly[i] = coeff de q^i
    binom = [1]
    for r in range(1, 25):
        binom.append(binom[-1] * (24 - r + 1) // r)          # C(24, r)
    factor = [binom[r] if r % 2 == 0 else -binom[r] for r in range(0, 25)]
    for m in range(1, N + 1):
        # multiplier par (1 - q^m)^24 : somme_{r} (-1)^r C(24,r) q^{m r}
        new = [0] * (N + 1)
        for i in range(0, N + 1):
            if poly[i] == 0:
                continue
            for r in range(0, 25):
                if i + m * r <= N:
                    new[i + m * r] += poly[i] * factor[r]
        poly = new
    return poly                    # poly[n-1] = tau(n) : Delta = q * prod, tau(n) = coeff de q^{n-1} du produit

N = 130
tau = tau_table(N)
valeurs_connues = {1: 1, 2: -24, 3: 252, 4: -1472, 5: 4830, 6: -6048,
                   7: -16744, 8: 84480, 9: -113643, 10: -115920, 11: 534612,
                   12: -370944, 13: -577738}
print("n    tau(n)      (Ramanujan 1916)")
for n in range(1, 15):
    ref = valeurs_connues.get(n, "")
    marque = "OK" if n in valeurs_connues and tau[n - 1] == valeurs_connues[n] else ""
    print(f"{n:<4} {tau[n - 1]:<12} {ref!s:<12} {marque}")
print()
print("Concordance avec les 13 valeurs publiees de Ramanujan :",
      sum(1 for n in valeurs_connues if tau[n - 1] == valeurs_connues[n]), "/", len(valeurs_connues))

n    tau(n)      (Ramanujan 1916)
1    1            1            OK
2    -24          -24          OK
3    252          252          OK
4    -1472        -1472        OK
5    4830         4830         OK
6    -6048        -6048        OK
7    -16744       -16744       OK
8    84480        84480        OK
9    -113643      -113643      OK
10   -115920      -115920      OK
11   534612       534612       OK
12   -370944      -370944      OK
13   -577738      -577738      OK
14   401856                    

Concordance avec les 13 valeurs publiees de Ramanujan : 13 / 13


### Lecture du résultat

Les treize premières valeurs calculées coïncident **exactement** avec les valeurs
publiées par Ramanujan en 1916 — y compris les signes (τ(7) = −16744). C'est la
garantie du niveau 1 : un exemple calculé, en arithmétique entière, sans arrondi.
Ce que ce niveau ne dit pas : si la suite obéit à une **structure**. C'est la
marche 2.

## 2. Marche 2 — la relation de Hecke, vérifiée sur la table

*(double [Langlands-01](Langlands/01-formes-modulaires-sl2z-hecke.ipynb), section valeurs propres de Hecke ; formalisée par `coeffHeckeT` dans le lac)*

Δ est une forme propre pour tous les opérateurs de Hecke : $T_p\,\Delta = \tau(p)\,\Delta$.
La formule des coefficients du lac ([marche 4](#4.-Marche-4-—-l'énoncé-dans-le-lac,-lu-depuis-le-fichier-réel))
donne alors, pour $k = 12$ :

$$\tau(p\,n) + p^{11}\,\tau(n/p) = \tau(p)\,\tau(n) \quad (n/p = 0 \text{ si } p \nmid n)$$

Cas particuliers : $\tau(p^2) = \tau(p)^2 - p^{11}$, puis la récurrence
$\tau(p^{r+1}) = \tau(p)\,\tau(p^r) - p^{11}\,\tau(p^{r-1})$. Vérifions sur la table.

In [2]:
# Marche 2 : la relation de Hecke verifiee sur la table calculee.
def t(n):
    return tau[n - 1] if 1 <= n <= N else None

print("relation        | membre gauche (calcule)      | membre droit (attendu)  | ecart")
ok = 0
total = 0
for p, rmax in ((2, 4), (3, 3), (5, 2)):
    for r in range(1, rmax + 1):
        n, np1, nm1 = p ** r, p ** (r + 1), p ** (r - 1)
        gauche = t(np1) + (p ** 11) * t(nm1)     # a(p^{r+1}) + p^11 a(p^{r-1})
        droite = t(p) * t(n)                     # a(p) a(p^r)
        total += 1
        ecart = gauche - droite
        ok += ecart == 0
        print(f"p={p:<2} r={r}         | {gauche:<28} | {droite:<23} | {ecart}")
print()
print(f"Relation de Hecke verifiee sur {ok}/{total} cas (p=2 r1-4, p=3 r1-3, p=5 r1-2)")

relation        | membre gauche (calcule)      | membre droit (attendu)  | ecart
p=2  r=1         | 576                          | 576                     | 0
p=2  r=2         | 35328                        | 35328                   | 0
p=2  r=3         | -2027520                     | -2027520                | 0
p=2  r=4         | -23691264                    | -23691264               | 0
p=3  r=1         | 63504                        | 63504                   | 0
p=3  r=2         | -28638036                    | -28638036               | 0
p=3  r=3         | -18466328160                 | -18466328160            | 0
p=5  r=1         | 23328900                     | 23328900                | 0
p=5  r=2         | -123161256750                | -123161256750           | 0

Relation de Hecke verifiee sur 9/9 cas (p=2 r1-4, p=3 r1-3, p=5 r1-2)


### Lecture du résultat

Neuf cas, neuf égalités **exactes** — et τ(16) = 987136 ou τ(27) = −73279080
émergent de la table sans avoir été saisis : c'est la récurrence de Hecke qui les
prédit. Niveau 2 atteint : l'identité est vérifiée là où elle est **discriminante**
(les grandes puissances de petits premiers). La question de la marche 3 : cette
relation caractérise-t-elle τ, ou n'importe quelle suite la satisfait-elle ?

## 3. Marche 3 — un témoin modifié viole la relation

Si la relation de Hecke était vide (satisfaite par toute suite), la vérifier ne
dirait rien. Prenons la suite perturbée la plus simple, $\tau'(n) = \tau(n) + n$,
et soumettons-la à la même épreuve.

In [3]:
# Marche 3 : la suite perturbee tau'(n) = tau(n) + n casse la relation de Hecke.
p, r = 2, 1
tp = lambda n: tau[n - 1] + n
gauche = tp(p ** (r + 1)) + p ** 11 * tp(p ** (r - 1))
droite = tp(p) * tp(p ** r)
print(f"tau' : p=2, r=1  ->  gauche = {gauche}, droit = {droite}, ecart = {gauche - droite}")
print()
viol = sum(1 for p2, rmax2 in ((2, 4), (3, 3), (5, 2)) for r2 in range(1, rmax2 + 1)
           if tp(p2 ** (r2 + 1)) + p2 ** 11 * tp(p2 ** (r2 - 1)) != tp(p2) * tp(p2 ** r2))
print(f"Sur les 9 cas de la marche 2, la suite perturbee en viole {viol}.")

tau' : p=2, r=1  ->  gauche = 2628, droit = 484, ecart = 2144

Sur les 9 cas de la marche 2, la suite perturbee en viole 9.


### Lecture du résultat

La suite perturbée viole la relation dès le premier cas — la relation de Hecke
**caractérise** : elle distingue τ de ses voisines arithmétiques. C'est ce qui
rend sa formalisation intéressante, et c'est exactement la sémantique que le lac
`hecke_lean` donne à `coeffHeckeT` : la définition du membre de gauche, pour une
suite quelconque — à charge pour le théorème de dire quand l'égalité tient.

## 4. Marche 4 — l'énoncé dans le lac, lu depuis le fichier réel

*(consomme [`hecke_lean/Hecke/HeckeOperator.lean`](hecke_lean/Hecke/HeckeOperator.lean) — port pédagogique de `anthropics/fermats-last-theorem`)*

Le lac ne se cite pas de mémoire : la cellule lit le fichier `.lean` réel du dépôt,
en extrait la définition `coeffHeckeT` — le membre de gauche de la relation — et
l'inventorie comme un consommateur le ferait.

In [4]:
# Marche 4 : lire les enonces du lac (le fichier reel, pas une copie).
import os
import re

CHEMIN = os.path.join("MyIA.AI.Notebooks", "SymbolicAI", "Lean", "hecke_lean", "Hecke", "HeckeOperator.lean")
texte = open(CHEMIN, encoding="utf-8").read() if os.path.exists(CHEMIN) else None
if texte is None:
    # appel depuis un autre cwd : chercher relatif au lake-root du depôt
    for racine in (".", "..", "../.."):
        c = os.path.join(racine, "MyIA.AI.Notebooks", "SymbolicAI", "Lean", "hecke_lean", "Hecke", "HeckeOperator.lean")
        if os.path.exists(c):
            CHEMIN, texte = c, open(c, encoding="utf-8").read()
            break

i = texte.index("def coeffHeckeT")
signature = texte[i: texte.index(":=", i)].strip()
corps = texte[i: texte.index("theorem", i)].strip()
doc_debut = texte.index("/-!")            # docstring du module
doc_module = texte[doc_debut + 4: texte.index("-/", doc_debut)][:300]

print("=== Docstring du module HeckeOperator.lean (extrait) ===")
print(doc_module.replace("\n", " ")[:280], "...")
print()
print("=== Definition formelle du membre de gauche ===")
print(corps[:260])
print()
# inventaire du lac : modules FR, theoremes par module
racine_lac = os.path.dirname(CHEMIN)
total_thm = 0
print("=== Inventaire du lac hecke_lean (modules FR) ===")
for f in sorted(os.listdir(racine_lac)):
    if f.endswith(".lean") and not f.endswith("_en.lean"):
        src = open(os.path.join(racine_lac, f), encoding="utf-8").read()
        nthm = len(re.findall(r"^\s*(?:theorem|lemma)\b", src, re.M))
        total_thm += nthm
        print(f"  {f:<22} {nthm:>3} enonces")
print(f"  {'TOTAL (FR)':<22} {total_thm:>3} enonces, miroirs _en non comptes")

=== Docstring du module HeckeOperator.lean (extrait) ===
# Opérateurs de Hecke classiques sur le demi-plan supérieur  Ce module introduit les opérateurs de Hecke « classiques » : pour un entier `p` (usuellement premier), l'opérateur `T_p` agit sur une fonction `f` du demi-plan supérieur ℍ par une somme finie d'actions de slash sur des  ...

=== Definition formelle du membre de gauche ===
def coeffHeckeT (k : ℤ) (p : ℕ) (a : ℕ → ℂ) (n : ℕ) : ℂ :=
  a (n * p) + if p ∣ n then (p : ℂ) ^ (k - 1) * a (n / p) else 0

/-- La formule des coefficients de la partie `U_p` : un simple
« échantillonnage » `a (n p)` de la suite. -/
def coeffHeckeU (p : ℕ) (a

=== Inventaire du lac hecke_lean (modules FR) ===
  DeltaTau.lean           10 enonces
  ElevenPid.lean           5 enonces
  FltRoute.lean           20 enonces
  HeckeOperator.lean      34 enonces
  SevenPid.lean            5 enonces


  ThirteenPid.lean        16 enonces
  TOTAL (FR)              90 enonces, miroirs _en non comptes


### Lecture du résultat

La définition `coeffHeckeT k p a n := a (n * p) + if p ∣ n then (p : ℂ) ^ (k - 1) * a (n / p) else 0`
est **mot pour mot** le membre de gauche de la marche 2 — avec la suite $a$
quelconque et le poids $k$ paramétrique : le lac formalise la formule, pas
seulement l'exemple. L'inventaire mesure la surface réelle : six modules français
(DeltaTau, ElevenPid, FltRoute, HeckeOperator, SevenPid, ThirteenPid) et leurs
miroirs `_en`, hors de ce décompte.

## 5. Marche 5 — la preuve, et ce qu'elle garantit

Un énoncé n'est pas une preuve. La cellule extrait le théorème de lecture
`coeffHeckeT_of_dvd` (le cas $p \mid n$) avec sa preuve, puis audite le module
sur les marqueurs interdits (`sorry`, `native_decide`) — le même audit que le
capstone [Lean-38](Lean-38-Capstone-Geometry-Lean-Python.ipynb) fait sur
`geometry_lean`.

In [5]:
# Marche 5 : la preuve du theoreme de lecture, et l'audit des marqueurs interdits.
i = texte.index("theorem coeffHeckeT_of_dvd")
extrait = texte[i: i + 320]
print("=== theoreme coeffHeckeT_of_dvd (fichier reel) ===")
print(extrait[:300])
print()
for marqueur in ("sorry", "native_decide"):
    n = len(re.findall(r"\b" + marqueur + r"\b", texte))
    print(f"audit HeckeOperator.lean : marqueur '{marqueur}' -> {n} occurrence(s)")

=== theoreme coeffHeckeT_of_dvd (fichier reel) ===
theorem coeffHeckeT_of_dvd (k : ℤ) {p n : ℕ} (h : p ∣ n) (a : ℕ → ℂ) :
    coeffHeckeT k p a n = a (n * p) + (p : ℂ) ^ (k - 1) * a (n / p) := by
  rw [coeffHeckeT, if_pos h]

/-- Lecture de `coeffHeckeT` dans le cas `p ∤ n` : seul subsiste le terme
d'échantillonnage `a (n p)`. -/
theorem coeffHeckeT

audit HeckeOperator.lean : marqueur 'sorry' -> 0 occurrence(s)
audit HeckeOperator.lean : marqueur 'native_decide' -> 0 occurrence(s)


### Lecture du résultat

La preuve tient en une réécriture (`rw [coeffHeckeT, if_pos h]`) : déplier la
définition puis résoudre l'alternative — la démonstration la plus courte possible
du fait que la définition et le théorème disent la même chose. L'audit compte
**zéro** `sorry` et **zéro** `native_decide` : l'énoncé lu est bien prouvé au
noyau. Niveau 4 atteint — et le pont final reste à traverser.

## 6. Marche 6 — Moonshine : les coefficients de j encodent le Monstre

*(double [Langlands-02](Langlands/02-monstrous-moonshine-invariant-j.ipynb))*

L'invariant modulaire $j = E_4^3/\Delta$ a la même mécanique de coefficients —
calculable par la même convolution entière ($E_4 = 1 + 240\sum \sigma_3(n) q^n$).
Mais ses coefficients font quelque chose d'étrange : ils se décomposent en
**sommes de dimensions de représentations irréductibles du groupe Monstre**
(l'ordre ~8·10⁵³ de Fischer–Griess). C'est la conjecture *Monstrous Moonshine*
(Conway–Norton 1979, démontrée par Borcherds 1992).

In [6]:
# Marche 6 : j = E4^3 / Delta par recursion entiere, puis les decompositions Monstre.
def sigma(n, k):
    return sum(d ** k for d in range(1, n + 1) if n % d == 0)

NJ = 6
e4 = [1] + [240 * sigma(n, 3) for n in range(1, NJ + 3)]
# E4^3 par convolution
e4c = [0] * (NJ + 3)
for a in range(len(e4)):
    for b in range(len(e4)):
        for c in range(len(e4)):
            if a + b + c <= NJ + 2:
                e4c[a + b + c] += e4[a] * e4[b] * e4[c]
# j : j*Delta = E4^3, recursivement (Delta = [0, 1, -24, 252, ...]).
# Coefficient de q^{n+1} : 1*delta[n+2] + sum_{m<=n} jeff[m]*delta[n+1-m] = e4c[n+1],
# et delta[0] = 0 fait tomber le terme m = n : chaque niveau determine jeff[n].
delta = [0] + tau[: NJ + 2]
jeff = [0] * (NJ + 1)          # jeff[i] = coeff de q^i ; le terme q^-1 vaut 1
for n in range(0, NJ + 1):
    lvl = n + 1
    s = e4c[lvl] - delta[lvl + 1]          # contribution du terme q^{-1} de j
    for m in range(0, n):
        s -= jeff[m] * delta[lvl - m]
    jeff[n] = s                             # division par delta[1] = 1
print("j(q) = 1/q +", " + ".join(str(c) for c in jeff[: NJ + 1]), "+ ...")
print()
dims_monstre = [1, 196883, 21296876, 842603213]
c1, c2 = jeff[1], jeff[2]
print(f"coefficient c1 = {c1}  -> 1 + 196883 = {1 + 196883}  | egalite : {c1 == 1 + 196883}")
print(f"coefficient c2 = {c2}  -> 1 + 196883 + 21296876 = {1 + 196883 + 21296876}  | egalite : {c2 == 1 + 196883 + 21296876}")
print()
print("Les deux premiers coefficients de j sont des sommes de dimensions de")
print("representations irreductibles du groupe Monstre (Moonshine, Conway-Norton 1979).")

j(q) = 1/q + 744 + 196884 + 21493760 + 864299970 + 20245856256 + 333202640600 + 4252023300096 + ...

coefficient c1 = 196884  -> 1 + 196883 = 196884  | egalite : True
coefficient c2 = 21493760  -> 1 + 196883 + 21296876 = 21493760  | egalite : True

Les deux premiers coefficients de j sont des sommes de dimensions de
representations irreductibles du groupe Monstre (Moonshine, Conway-Norton 1979).


### Lecture du résultat

$196884 = 1 + 196883$ et $21493760 = 1 + 196883 + 21296876$ : les dimensions des
plus petites représentations du Monstre, additionnées, redonnent exactement les
coefficients de $j$ — recalculés ici depuis le produit $E_4^3/\Delta$, pas recopiés.
La « coïncidence monstrueuse » de McKay est une **sortie calculée**. Le pont de la
sous-série est franchi : analyse (Δ), arithmétique (Hecke), algèbre (Monstre) —
les trois volets du README.

## 7. Les garanties, côte à côte

| Marche | Geste | Ce qui est gagné | Ce qui reste hors de portée |
|---|---|---|---|
| 1 — table | convolution entière de Δ | 13 valeurs de Ramanujan exactes | aucune structure |
| 2 — Hecke | identité vérifiée p = 2, 3, 5 | la relation tient où elle discrimine | une preuve pour tout p, tout n |
| 3 — témoin | suite perturbée τ + n | la relation **caractérise** τ | — |
| 4 — lac | `coeffHeckeT` lu au fichier | la formule est formalisée, suite quelconque | la preuve |
| 5 — preuve | `coeffHeckeT_of_dvd` + audit | démontrée au noyau, 0 `sorry` | les formes elles-mêmes (hors périmètre du lac) |
| 6 — Monstre | $E_4^3/\Delta$ et les dimensions | Moonshine recalculé, pas recopié | la correspondance complète (Borcherds) |

## 8. La sous-série et ses voisins

| Où | Quoi | Rôle dans l'escalier |
|---|---|---|
| [Langlands-01](Langlands/01-formes-modulaires-sl2z-hecke.ipynb) | formes modulaires de SL₂(ℤ), opérateurs de Hecke, pont Γ₀(11) ↔ courbe 11a1 | la marche 1–2 en profondeur |
| [Langlands-02](Langlands/02-monstrous-moonshine-invariant-j.ipynb) | l'invariant j et le groupe Monstre | la marche 6 en profondeur |
| [README de la sous-série](Langlands/README.md) | le geste (identités calculées), conventions, sources primaires (Serre, Borcherds) | la charte |
| [lac `hecke_lean`](hecke_lean/README.md) | opérateurs de Hecke, courbe de Frey — port de `anthropics/fermats-last-theorem` | les marches 4–5 |
| [Lean-29](Lean-29-Hecke-Operators-Native.ipynb) | compagnon **natif** du lac (kernel `lean4-wsl`) | le lac *exécuté*, pour qui veut monter plus haut |
| [Lean-37](Lean-37-Capstone-Serre100.ipynb) | capstone de la sous-série sœur Serre100 | le même geste escalier, autre famille |

## 9. Exercices

Trois marches à refaire soi-même. Les cellules s'exécutent telles quelles (C.1) ;
à vous de remplacer les `result = None` par la comparaison demandée.

### Exercice 1 — la relation de Hecke aux premiers 11 et 13

La table `tau` va jusqu'à N = 130 : elle couvre τ(121) mais pas τ(13²) = τ(169). Recalculez
la table à N = 180 (une ligne), puis vérifiez `tau(p^2) == tau(p)^2 - p^11` pour
p = 11 et p = 13 — les deux premiers après la marche 2.

In [7]:
# Exercice 1 : Hecke aux premiers 11 et 13 (table etendue a N = 180).
# Etape 1 : recalculer la table.
tau_ext = tau_table(180)
# Etape 2 : pour p = 11 et p = 13, comparer tau_ext[p^2 - 1] et tau_ext[p - 1]**2 - p**11.
resultat_11 = None  # TODO etudiant : resultat_11 = (tau_ext[121-1], tau_ext[11-1]**2 - 11**11)
resultat_13 = None  # TODO etudiant : idem pour p = 13
print("tau(11) =", tau_ext[10], "| tau(13) =", tau_ext[12])
print("Exercice a completer : poser resultat_11 et resultat_13 (paires attendu vs recursif).")

tau(11) = 534612 | tau(13) = -577738
Exercice a completer : poser resultat_11 et resultat_13 (paires attendu vs recursif).


### Exercice 2 — le deuxième coefficient et le Monstre

Le coefficient $c_2$ de $j$ vaut $21493760$. Avec les quatre premières dimensions
de représentations du Monstre — `[1, 196883, 21296876, 842603213]` — cherchez une
combinaison à petits multiplicateurs (chacun ≤ 3) qui redonne $c_3$. *Indice : la
décomposition existe et est unique avec ces bornes.*

In [8]:
# Exercice 2 : decomposer 21493760 en somme a petits multiplicateurs des dims Monstre.
from itertools import product

dims = [1, 196883, 21296876, 842603213]
cible = 21493760
# Etape : parcourir les multiplicateurs (m0, m1, m2, m3) dans {0..3}^4 et garder la somme egale.
decomposition = None  # TODO etudiant : la boucle product(range(4), repeat=4) et le test
print("Exercice a completer : chercher (m0..m3) tel que sum(m*d) == cible.")

Exercice a completer : chercher (m0..m3) tel que sum(m*d) == cible.


### Exercice 3 — auditer tout le lac

La marche 5 audite un module. Étendez l'audit aux six modules français du lac :
pour chacun, compter `sorry` et `native_decide`. *Le résultat attendu : zéro
partout — c'est ce que « socle formel » veut dire.*

In [9]:
# Exercice 3 : audit des six modules FR du lac hecke_lean.
import re

def audit_module(chemin):
    src = open(chemin, encoding="utf-8").read()
    counts = {}
    # TODO etudiant : pour chaque marqueur ("sorry", "native_decide"), compter les
    # occurrences par mot entier (re.findall avec \b) et remplir counts.
    return counts

print("Exercice a completer : boucle sur les fichiers FR de hecke_lean/Hecke/ et bilan.")

Exercice a completer : boucle sur les fichiers FR de hecke_lean/Hecke/ et bilan.


## 10. Conclusion — ce que l'escalier laisse au lecteur

Une identité, six garanties : une table exacte, une relation vérifiée là où elle
discrimine, un témoin qui prouve qu'elle caractérise, une définition formalisée,
une preuve auditée, et un pont vers le Monstre. L'escalier ne remplace pas la
sous-série — il dit **par où y entrer** :

- pour la théorie (SL₂(ℤ), Γ₀(11), la correspondance) : [Langlands-01](Langlands/01-formes-modulaires-sl2z-hecke.ipynb) ;
- pour Moonshine et le groupe Monstre : [Langlands-02](Langlands/02-monstrous-moonshine-invariant-j.ipynb) ;
- pour le lac exécuté dans un vrai kernel Lean : [Lean-29](Lean-29-Hecke-Operators-Native.ipynb) ;
- pour la direction orthogonale — la marée de Grothendieck contre les voies
  surprenantes : [Lean-15](Lean-15-Grothendieck-Tribute.ipynb), section « Les
  limites de la marée ».